In [1]:
!pip install tldextract Levenshtein python-whois

# 하는 일: 링크를 쪼개고, 글자 차이를 세고, 도메인 정보를 조회하는 도구를 설치함
# 이유: 이 기능들이 파이썬 기본에는 없기 때문

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.0 MB/s eta 0:00:00


In [2]:
import tldextract

ext = tldextract.extract("http://naver-login.com/verify")
print(ext.subdomain, ext.domain, ext.suffix)

# 하는 일: 주소를 앞부분(subdomain) / 이름(domain) / 끝(suffix)으로 나눔
# 이유: 진짜 이름 부분만 골라야 유명 이름을 따라 했는지 비교할 수 있음

 naver-login com


In [3]:
import Levenshtein

print(Levenshtein.distance("naver-login", "naver"))
print(Levenshtein.distance("taiegram", "telegram"))

# 하는 일: 한 이름을 다른 이름으로 바꾸려면 글자를 몇 개 고쳐야 하는지 셈 (작을수록 비슷함)
# 이유: telegram → taiegram처럼 글자만 살짝 바꾼 가짜 주소를 잡기 위해

6
2


In [4]:
import whois

try:
    w = whois.whois("google.com")
    print(w.creation_date)
except Exception as e:
    print("조회 실패:", e)

# 하는 일: 인터넷 등록 장부에서 도메인이 언제 만들어졌는지 날짜를 가져옴
# 이유: 얼마 전에 급하게 만든 도메인일수록 피싱일 가능성이 높기 때문
# 참고: 날짜가 목록으로 나올 수 있음. 조회가 실패하는 것도 정상이라 try/except로 감쌈

[datetime.datetime(1997, 9, 15, 4, 0, tzinfo=tzoffset('UTC', 0)), datetime.datetime(1997, 9, 15, 7, 0, tzinfo=tzoffset('UTC', 0))]


In [5]:
import json
import datetime
import tldextract
import Levenshtein
import whois

# 진짜 이름 목록 (사례조사에서 나온 것부터 시작. 나중에 늘리기)
BRANDS = ["naver", "telegram", "kakao", "google"]

def analyze_url_sample(url, use_whois=True):
    reasons = []        # 점수가 나온 이유를 모아 두는 목록
    unavailable = []    # 확인하지 못한 항목을 모아 두는 목록

    ext = tldextract.extract(url)
    name = ext.domain.lower()
    # 하는 일: 주소에서 이름 부분만 뽑음 (셀 2에서 연습한 것)

    # 1. 도메인 유사 (최대 35점)
    for brand in BRANDS:
        if name != brand and (brand in name or Levenshtein.distance(name, brand) <= 2):
            reasons.append({
                "factor": "도메인 유사",
                "points": 35,
                "text": f"유명 이름('{brand}')과 비슷하지만 똑같지는 않은 도메인입니다."
            })
            break
    # 하는 일: 진짜 이름이 들어 있거나 글자가 1~2개만 다르면 점수를 줌 (셀 3에서 연습한 것)
    # 이유: 진짜 이름과 똑같으면 진짜 사이트이므로 점수를 주지 않음 (name != brand)

    # 2. 키워드 (최대 10점)
    if "verify" in url.lower() or "login" in url.lower():
        reasons.append({
            "factor": "키워드",
            "points": 10,
            "text": "주소에 verify, login처럼 인증을 유도하는 단어가 들어 있습니다."
        })
    # 하는 일: 주소에 인증을 유도하는 단어가 있으면 점수를 줌
    # 이유: 피싱 주소에는 로그인이나 인증을 하라는 단어가 자주 들어가기 때문

    # 3. 도메인 생성일 (최대 30점): 만든 지 30일 미만이면 점수
    if use_whois:
        try:
            if not ext.suffix:
                raise ValueError("도메인 형태가 아님")
            info = whois.whois(f"{ext.domain}.{ext.suffix}")
            created = info.creation_date
            if isinstance(created, list):      # 날짜가 목록으로 나오면 첫 번째 것을 씀
                created = created[0]
            if created is None:
                raise ValueError("생성일 정보 없음")
            if created.tzinfo is None:         # 시간대 정보가 없으면 UTC로 맞춤
                created = created.replace(tzinfo=datetime.timezone.utc)
            days = (datetime.datetime.now(datetime.timezone.utc) - created).days
            if days < 30:
                reasons.append({
                    "factor": "생성일",
                    "points": 30,
                    "text": f"도메인이 만들어진 지 {days}일밖에 안 됐습니다."
                })
        except Exception:
            unavailable.append("whois")
    else:
        unavailable.append("whois")
    # 하는 일: 도메인이 만들어진 지 얼마나 됐는지 조회함 (셀 4에서 연습한 것)
    # 이유: 조회에 실패하면 0점이 아니라 unavailable에 적어서 "확인 못 했다"는 걸 구분함

    # 4. 점수 합치기
    score = sum(r["points"] for r in reasons)
    # 하는 일: 이유별 점수를 모두 더함
    # 이유: README 약속 1번 "score는 reasons의 points를 더한 값"

    # 5. 팀 약속 모양으로 돌려주기
    return {
        "type": "url",
        "input": url,
        "score": score,
        "model_probability": None,   # 아직 모델이 없어서 비어 있음 (JSON에서는 null이 됨)
        "reasons": reasons,
        "unavailable": unavailable
    }
    # 이유: A가 만든 결과를 B(화면)와 C(장부)가 같은 칸 이름으로 받아 쓰게 하려고

# 실행해 보기 (출력된 링크는 누르지 않기)
result = analyze_url_sample("http://naver-login.com/verify")
print(json.dumps(result, ensure_ascii=False, indent=2))

ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


{
  "type": "url",
  "input": "http://naver-login.com/verify",
  "score": 45,
  "model_probability": null,
  "reasons": [
    {
      "factor": "도메인 유사",
      "points": 35,
      "text": "유명 이름('naver')과 비슷하지만 똑같지는 않은 도메인입니다."
    },
    {
      "factor": "키워드",
      "points": 10,
      "text": "주소에 verify, login처럼 인증을 유도하는 단어가 들어 있습니다."
    }
  ],
  "unavailable": []
}


In [6]:
# 확인 1: naver-login.com의 생성일이 실제로 어떻게 나오는지 보기 (등록 정보만 조회하고 접속은 하지 않음)
try:
    info = whois.whois("naver-login.com")
    print("생성일:", info.creation_date)
except Exception as e:
    print("조회 실패:", e)

# 확인 2: 진짜 이름과 철자만 바꾼 이름이 잘 구분되는지 (whois는 건너뜀)
tests = ["http://google.com/", "http://taiegram.com/login"]
for u in tests:
    r = analyze_url_sample(u, use_whois=False)
    print(u, "→ 점수", r["score"], "| 이유", [x["factor"] for x in r["reasons"]], "| 확인 못 함", r["unavailable"])

# 하는 일: 위 추측이 맞는지 보고, 진짜 이름은 점수 0, 철자 바꾼 이름은 점수가 붙는지 확인함
# 이유: 점수 함수가 "진짜는 안전, 가짜는 위험"으로 나뉘는지 시험하기 위해

ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


생성일: 2026-03-17 00:41:38+00:00
http://google.com/ → 점수 0 | 이유 [] | 확인 못 함 ['whois']
http://taiegram.com/login → 점수 45 | 이유 ['도메인 유사', '키워드'] | 확인 못 함 ['whois']


In [7]:
# ===== 셀 7: 샘플 2 주소를 샘플 1 함수에 넣어 보기 =====
url2 = "http://gov-support.example/apply"

print(tldextract.extract(url2))
# 하는 일: 주소를 앞부분 / 이름 / 끝으로 나눈 결과를 출력함 (셀 2에서 연습한 것)
# 이유: 함수가 이 주소의 어느 부분을 "이름"으로 보는지 확인하려고

before = analyze_url_sample(url2)
print("점수:", before["score"], "| 이유:", [r["factor"] for r in before["reasons"]], "| 확인 못 함:", before["unavailable"])
# 하는 일: 샘플 1 함수에 샘플 2 주소를 넣고 점수, 이유 이름, 확인 못 한 항목을 한 줄로 출력함
# 이유: 샘플 2를 만들기 전에 지금 함수가 정부기관 사칭 주소를 어떻게 판단하는지 시험하기 위해
# 참고: .example은 연습용 가짜 주소라 인터넷 등록 장부(whois)에 없음 → 조회가 항상 실패함
#       → "조회 실패" 상황을 매번 똑같이 만들 수 있어서 샘플 2에 딱 맞음

ExtractResult(subdomain='gov-support', domain='example', suffix='', is_private=False)
점수: 0 | 이유: [] | 확인 못 함: ['whois']


In [8]:
# ===== 셀 8: 샘플 1 함수 + 검사 3개 = 새 함수 (analyze_url_v2) =====
GOV_WORDS = ["gov", "minwon"]
# 하는 일: 공공기관을 뜻하는 단어 목록을 만듦 (gov = government(정부), minwon = 민원)
# 이유: 정부기관을 사칭하는 피싱 주소를 잡기 위해. 사례조사하면서 단어를 늘리기

def is_real_gov(ext):
    return ext.suffix == "go.kr" or (ext.domain == "gov" and ext.suffix == "kr")
# 하는 일: 주소 끝이 진짜 정부 주소인지 확인함 (예: minwon.go.kr, www.gov.kr이면 True)
# 이유: 우리나라 정부기관 주소는 .go.kr로 끝나고, 정부24는 gov.kr을 씀. 이 둘은 진짜로 봄

def analyze_url_v2(url):
    result = analyze_url_sample(url)
    # 하는 일: 먼저 샘플 1 함수(셀 5)를 그대로 실행해서 결과를 받아 옴
    # 이유: 완성한 셀 5를 건드리지 않고, 그 위에 검사만 덧붙이려고

    ext = tldextract.extract(url)
    labels = [p.lower() for p in [ext.domain] + ext.subdomain.split(".") if p]
    factors = [r["factor"] for r in result["reasons"]]
    # 하는 일: 주소를 다시 나누고, 이름 부분 + 앞부분을 한 목록(labels)에 모으고,
    #          셀 5에서 이미 나온 이유들의 이름(factors)도 모아 둠
    #          예: gov-support.example → labels = ["example", "gov-support"]
    # 이유: 셀 7에서 본 것처럼 셀 5는 이름 부분(example)만 봐서 앞부분에 숨긴 이름을 놓치기 때문

    # 추가 검사 1: 도메인 유사 (35점) - 앞부분에 숨긴 유명 이름·기관 이름까지 확인
    if "도메인 유사" not in factors and ext.domain.lower() not in BRANDS and not is_real_gov(ext):
        for word in BRANDS + GOV_WORDS:
            if any(word in label for label in labels):
                if word in GOV_WORDS:
                    text = f"정부기관 이름('{word}')을 따라 했지만 진짜 정부 주소가 아닙니다."
                else:
                    text = f"유명 이름('{word}')과 비슷하지만 똑같지는 않은 도메인입니다."
                result["reasons"].insert(0, {"factor": "도메인 유사", "points": 35, "text": text})
                break
    # 하는 일: 셀 5에서 도메인 유사를 못 찾았고 진짜 사이트도 아니면,
    #          주소 어디에든 유명 이름이나 기관 이름이 들어 있는지 봄
    # 이유: kakao-pay.example, gov-support.example처럼 이름을 앞부분에 숨긴 주소를 잡기 위해
    # 참고: insert(0, ...)는 목록 맨 앞에 넣는다는 뜻. 샘플 1처럼 도메인 유사가 첫 번째 이유가 되게 함

    # 추가 검사 2: 기관명 불일치 (15점)
    if not is_real_gov(ext) and any(w in label for w in GOV_WORDS for label in labels):
        result["reasons"].append({
            "factor": "기관명 불일치",
            "points": 15,
            "text": "공공기관 이름을 내세웠지만 주소 끝이 정부 공식 주소(.go.kr)가 아닙니다."
        })
    # 하는 일: 주소에 공공기관 단어(gov, minwon)가 있는데 끝이 진짜 정부 주소가 아니면 점수를 줌
    # 이유: "정부 지원금 신청" 같은 피싱은 기관 이름만 빌리고 주소는 아무 데나 만들기 때문
    # 참고: 도메인 유사는 "이름을 따라 했다", 기관명 불일치는 "공식 주소가 아니다"를 보는 것.
    #       gov 하나로 두 항목이 같이 걸릴 수 있음 → 중복 점수인지는 팀 논의 포인트

    # 추가 검사 3: URL 구조 (10점)
    host = ".".join(p for p in [ext.subdomain, ext.domain, ext.suffix] if p)
    if "-" in host:
        result["reasons"].append({
            "factor": "URL 구조",
            "points": 10,
            "text": "하이픈(-)이 들어간 주소입니다. 기관이나 회사 이름에 단어를 이어 붙여 꾸몄을 가능성이 있습니다."
        })
    # 하는 일: 사이트 이름 부분(host)에 하이픈(-)이 있으면 점수를 줌
    # 이유: gov-support, naver-login처럼 진짜 이름에 단어를 이어 붙인 가짜 주소가 많기 때문
    # 참고: 정상 사이트도 하이픈을 쓰는 경우가 있어서 점수를 10점으로 낮게 둠. 기준은 팀 논의 후 조정

    result["score"] = sum(r["points"] for r in result["reasons"])
    return result
    # 하는 일: 이유가 늘었으니 점수를 다시 더해서 돌려줌
    # 이유: README 약속 1번 "score는 reasons의 points를 더한 값"

print("셀 8 준비 완료")

셀 8 준비 완료


In [9]:
# ===== 셀 9: 샘플 2 결과 만들기 + 약속 확인 =====
sample2 = analyze_url_v2(url2)
print(json.dumps(sample2, ensure_ascii=False, indent=2))
# 하는 일: 새 함수로 샘플 2 주소를 분석하고 결과를 보기 좋게 출력함
# 이유: 샘플을 손으로 쓰지 않고 함수가 실제로 만든 결과로 만들어야 함수와 샘플 모양이 항상 같음

assert sample2["score"] == sum(r["points"] for r in sample2["reasons"])
assert "whois" in sample2["unavailable"]
assert all(r["factor"] != "생성일" for r in sample2["reasons"])
print("약속 확인 통과!")
# 하는 일: assert는 "이게 맞아야 한다"는 검사. 틀리면 빨간 AssertionError가 나고 멈춤
# 확인 1: score가 points 합과 같은지 (README 약속 1번)
# 확인 2: 조회 실패한 whois가 unavailable에 적혔는지 (README 약속 3번)
# 확인 3: 확인 못 한 생성일이 0점짜리 이유로 끼어들지 않았는지 (README 약속 3번)

for u in ["http://google.com/", "http://taiegram.com/login", "https://www.gov.kr/", "https://www.minwon.go.kr/"]:
    print(u, "→ 점수", analyze_url_v2(u)["score"])
# 하는 일: 진짜 사이트와 셀 6에서 시험한 주소를 새 함수에도 넣어 봄
# 이유: 검사를 덧붙이다가 진짜 정부 사이트까지 위험하다고 잘못 판단하지 않는지 확인하려고

{
  "type": "url",
  "input": "http://gov-support.example/apply",
  "score": 60,
  "model_probability": null,
  "reasons": [
    {
      "factor": "도메인 유사",
      "points": 35,
      "text": "정부기관 이름('gov')을 따라 했지만 진짜 정부 주소가 아닙니다."
    },
    {
      "factor": "기관명 불일치",
      "points": 15,
      "text": "공공기관 이름을 내세웠지만 주소 끝이 정부 공식 주소(.go.kr)가 아닙니다."
    },
    {
      "factor": "URL 구조",
      "points": 10,
      "text": "하이픈(-)이 들어간 주소입니다. 기관이나 회사 이름에 단어를 이어 붙여 꾸몄을 가능성이 있습니다."
    }
  ],
  "unavailable": [
    "whois"
  ]
}
약속 확인 통과!
http://google.com/ → 점수 0
http://taiegram.com/login → 점수 45
https://www.gov.kr/ → 점수 0
https://www.minwon.go.kr/ → 점수 0


In [10]:
# ===== 셀 10: 샘플 2를 파일로 저장하고 내 컴퓨터로 내려받기 =====
from google.colab import files

with open("url_whois_fail.json", "w", encoding="utf-8") as f:
    json.dump(sample2, f, ensure_ascii=False, indent=2)
files.download("url_whois_fail.json")
# 하는 일: 결과를 url_whois_fail.json 파일로 저장한 뒤 다운로드 폴더로 내려받음
# ensure_ascii=False: 한글이 \uc0d8 같은 이상한 글자로 바뀌지 않게 함
# indent=2: 사람이 읽기 좋게 줄 바꾸고 들여쓰기 함
# 주의: 다운로드 창이 안 뜨면 주소창 오른쪽 "팝업 차단" 아이콘에서 허용

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>